In [1]:
from huggingface_hub import notebook_login
import os
notebook_login()
repo_name='jkhyjkhy/wav2vec2-large-xlsr-sw-ASR'

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE = '/content/drive/MyDrive/ASR_wav2vec_project/preprocessing/processed_data/14.0-delta-2023-06-23'
CSV = f'{BASE}/manifest_sw_14_0_delta.csv'
AUDIO_DIR = f'{BASE}/cleaned_sw_audio_14_0_delta'
# If you're using a pre-configured HuggingFace processor, you can comment out or remove VOCAB
VOCAB = f'{BASE}/vocab.json'

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(CSV)
df["wav_path"] = df["wav_path"].apply(lambda p: os.path.join(AUDIO_DIR, os.path.basename(p)))
print(f"Total samples in dataset: {len(df)}")

# Filter out missing audio files
df['file_exists'] = df['wav_path'].apply(os.path.exists)
missing_count = len(df[~df['file_exists']])
print(f"Missing audio files: {missing_count}")

df = df[df['file_exists']].drop(columns=['file_exists'])
print(f"Samples after filtering: {len(df)}")

In [5]:
from datasets import Dataset, Audio
dataset = Dataset.from_pandas(df[['wav_path', 'transcript']])

In [6]:
import re

def remove_special_characters(batch, column_names='transcript'):
  chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\(\)]'
  batch[column_names] = re.sub(chars_to_ignore_regex, '', batch[column_names]).lower() + " "
  return batch

dataset = dataset.map(remove_special_characters)

print(len(dataset['wav_path']))

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

270


In [9]:
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("jkhyjkhy/wav2vec2-large-xlsr-sw-ASR")
model = AutoModelForCTC.from_pretrained("jkhyjkhy/wav2vec2-large-xlsr-sw-ASR")

preprocessor_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/598 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

In [11]:
import torch
def map_to_result(batch):
  with torch.no_grad():
    input_values = torch.tensor(batch["input_values"]).unsqueeze(0)
    logits = model(input_values).logits

  pred_ids = torch.argmax(logits, dim=-1)
  print(pred_ids)
  batch["pred_str"] = processor.batch_decode(pred_ids)[0]
  batch["text"] = processor.decode(batch["labels"], group_tokens=False)

  return batch

In [12]:
dataset = dataset.cast_column("wav_path", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["wav_path"]

    audio_inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"])

    batch["input_values"] = audio_inputs["input_values"][0]
    batch["input_length"] = len(batch["input_values"])
    label_inputs = processor.tokenizer(batch["transcript"])
    batch["labels"] = label_inputs["input_ids"]

    return batch

In [17]:
dataset.column_names
embeded_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

AttributeError: 'Dataset' object has no attribute 'keys'

In [21]:
embeded_dataset.column_names

['input_values', 'input_length', 'labels']

In [ ]:
!pip install evaluate jiwer bert_score sacrebleu

In [ ]:
import evaluate

# Load metrics
wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')
bleu_metric = evaluate.load('bleu')
bertscore_metric = evaluate.load('bertscore')

# SER (Sentence Error Rate) - custom function
def compute_ser(predictions, references):
    """Calculate Sentence Error Rate - percentage of sentences with any error"""
    errors = sum(1 for p, r in zip(predictions, references) if p.strip() != r.strip())
    return errors / len(references)

# Levenshtein Distance - custom function  
def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def compute_avg_levenshtein(predictions, references):
    """Calculate average Levenshtein distance across all samples"""
    distances = [levenshtein_distance(p, r) for p, r in zip(predictions, references)]
    return np.mean(distances), distances

In [ ]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Basic metrics
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    ser = compute_ser(pred_str, label_str)
    
    # Levenshtein
    avg_lev, _ = compute_avg_levenshtein(pred_str, label_str)

    return {
        "wer": wer, 
        "cer": cer,
        "ser": ser,
        "levenshtein": avg_lev
    }

In [ ]:
results = embeded_dataset.map(map_to_result, remove_columns=embeded_dataset.column_names)

predictions = results["pred_str"]
references = results["text"]

# 1. WER (Word Error Rate)
wer_score = wer_metric.compute(predictions=predictions, references=references)

# 2. CER (Character Error Rate)
cer_score = cer_metric.compute(predictions=predictions, references=references)

# 3. SER (Sentence Error Rate)
ser_score = compute_ser(predictions, references)

# 4. BLEU Score
# BLEU expects: predictions as strings, references as list of strings
bleu_refs = [[ref] for ref in references]  # each reference wrapped in a list
bleu_score = bleu_metric.compute(predictions=predictions, references=bleu_refs)

# 5. BERTScore
bertscore_results = bertscore_metric.compute(
    predictions=predictions, 
    references=references, 
    lang="sw"  # Swahili
)
bertscore_precision = np.mean(bertscore_results['precision'])
bertscore_recall = np.mean(bertscore_results['recall'])
bertscore_f1 = np.mean(bertscore_results['f1'])

# 6. Levenshtein Distance
avg_levenshtein, all_levenshtein = compute_avg_levenshtein(predictions, references)

# Print Results
print("="*60)
print("                    Evaluation Results")
print("="*60)
print(f"\n[Edit Distance Based Metrics]")
print(f"  WER (Word Error Rate):        {wer_score:.4f} ({wer_score*100:.2f}%)")
print(f"  CER (Character Error Rate):   {cer_score:.4f} ({cer_score*100:.2f}%)")
print(f"  SER (Sentence Error Rate):    {ser_score:.4f} ({ser_score*100:.2f}%)")
print(f"  Avg Levenshtein Distance:     {avg_levenshtein:.2f} chars")

print(f"\n[N-gram Based Metrics]")
print(f"  BLEU Score:                   {bleu_score['bleu']:.4f} ({bleu_score['bleu']*100:.2f}%)")

print(f"\n[Semantic Similarity Metrics]")
print(f"  BERTScore Precision:          {bertscore_precision:.4f}")
print(f"  BERTScore Recall:             {bertscore_recall:.4f}")
print(f"  BERTScore F1:                 {bertscore_f1:.4f}")

print("="*60)

In [28]:
from IPython.display import display, HTML
import random
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

In [29]:
show_random_elements(results)

,pred_str,text
0,hivi lee zinaendelea kutoa nafasi kisalama kuwasaidia watoto wapike na wanawake kujujana,hivi leo kinaendelea kutoa nafasi salama kuwasaidia watoto wa kike na wanawake vijana
1,mifugo inaweza kuathiriwa na ugonjwa huu wa upele,mifugo inaweza kuathiriwa na ugonjwa huu wa upele
2,weka safuria yako jikoni,weka sufuria yako jikoni
3,kanda unga na ti mafuta kiasi,kanda unga na tia mafuta kiasi
4,mnyama asiye mgonjwa badaa anapoonyawadamu wingine,mnyama asiye mgonjwa baadaye anapomnyonya damu mnyama mwingine
5,alisema jambo muhimu ni kuchukua msima modhabiti na kuzuia manyanyaso dhidi ya raia,alisema jambo muhimu ni kuchukua msimamo thabiti na kuzuia manyanyaso dhidi ya raia
6,katika siku ya pili ya vikao vya seneti kuhusu uteuzi wake katika maakama ya ju jaji katanji braun jakson alitabiliwa na maswali kwa saa kadha,katika siku ya pili ya vikao vya seneti kuhusu uteuzi wake katika mahakama ya juu jaji ketanji brown jackson alikabiliwa na maswali kwa saa kadhaa
7,ugonjwa wa miguu na midomo,ugonjwa wa miguu na midomo
8,huwa kwa nadra au mara chache la kisababishia nimbako,huua kwa nadra au mara chache lakini husababisha mimba kutoka
9,matumizi ya viazilisha,matumizi ya viazi lishe
